# 真实数据案例：基因调控网络推断与共表达分析

来自《生物医学网络科学》（Biomedical Network Science）。数据文件放在与本笔记本同级的 `data/` 文件夹中。

# 真实数据案例：乳腺癌活检的加权基因共表达网络

**问题**：在 103 例乳腺癌化疗前活检中，基因能否压缩成少数几个共表达模块？哪个模块与雌激素受体（ER）状态关联最强，它的 hub 基因是谁？共表达边与已知的蛋白物理相互作用重叠多少，偏相关能否把重叠提高？

**数据来源**

- 表达谱：NCBI GEO 数据集 GDS4057（系列 GSE22093，Affymetrix HG-U133A），HER2 阴性乳腺癌化疗前活检 103 例，其中 ER（免疫组化）阳性 42 例、阴性 56 例、缺失 5 例。原文：Iwamoto T, Bianchini G, et al. *J Natl Cancer Inst* 2011;103:264–272。NCBI 对 GEO 数据的使用和再分发不设限制。为控制文件大小，只保留了表达量不低于中位数的基因中方差最大的 1000 个（处理步骤见 `data/SOURCES.md`）。
- 参照网络：STRING v12.0 人类物理相互作用子网络，综合分数 ≥ 700（CC BY 4.0；Szklarczyk D, et al. *Nucleic Acids Res* 2023;51:D638–D646）。

分析步骤按 [WGCNA](gene-network-inference.wgcna.html) 的标准流程：选软阈值幂次 → 拓扑重叠 → 层次聚类切树 → 模块特征基因与性状关联 → hub 基因；随后用 [推断结果的评估](gene-network-inference.evaluation.html) 中的 AUPR 把共表达边、收缩偏相关与 STRING 物理相互作用比较。为了在浏览器里也能运行，这里用 numpy/scipy 自己实现各步，切树用固定高度代替动态剪切。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "PingFang SC", "Heiti SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

expr = pd.read_csv("data/breast_gds4057_expr_top1000.csv", index_col="gene")
samples = pd.read_csv("data/breast_gds4057_samples.csv", index_col="sample")
print("表达矩阵（基因 × 样本）：", expr.shape)
print(samples["er_ihc"].value_counts().to_dict())

## 1 读入与清洗

WGCNA 的 FAQ 建议先检查离群样本。这里用样本间相关的平均值找离群样本：若某个样本与其他样本的平均相关比全体低 3 个标准差以上，就剔除。随后把每个基因在样本间标准化。ER 缺失的 5 例保留在网络构建中（构网不需要性状），只在模块–性状关联时剔除。

In [ ]:
X = expr.T.loc[samples.index]                    # 样本 × 基因
cs = np.corrcoef(X.values)                       # 样本 × 样本相关
np.fill_diagonal(cs, np.nan)
mean_c = np.nanmean(cs, axis=1)
z_out = (mean_c - mean_c.mean()) / mean_c.std()
keep = z_out > -3
print("样本平均相关范围：%.3f – %.3f；剔除离群样本 %d 个" % (mean_c.min(), mean_c.max(), (~keep).sum()))
X = X.loc[keep]
Z = (X - X.mean()) / X.std(ddof=0)
n, p = Z.shape
genes = np.array(Z.columns)
print("用于构网：%d 个样本、%d 个基因" % (n, p))

## 2 构网：有符号软阈值与无标度拟合

有符号网络 $a_{ij}=\left(\frac{1+r_{ij}}{2}\right)^{\beta}$ 只把同向变化的基因连起来。对 $\beta = 2,4,\dots,20$ 计算无标度拟合指数（$\log_{10}p(k)$ 对 $\log_{10}k$ 回归的 $R^2$ 乘以斜率的负号），选首次达到 0.8 的最小 $\beta$。有符号网络通常需要的 $\beta$ 比无符号网络大（无符号 $|r|^\beta$ 中弱相关压得更快）。

In [ ]:
Rg = np.corrcoef(Z.values.T)                     # 基因 × 基因相关

def sft_fit(A, nbins=10):
    k = A.sum(axis=1) - 1.0                      # 去掉对角线 a_ii = 1
    cnt, edges = np.histogram(k, bins=nbins)
    mid = (edges[:-1] + edges[1:]) / 2
    ok = cnt > 0
    res = stats.linregress(np.log10(mid[ok]), np.log10(cnt[ok] / cnt.sum()))
    return -np.sign(res.slope) * res.rvalue ** 2, k.mean()

betas = list(range(2, 21, 2))
fit = [sft_fit(((1 + Rg) / 2) ** b) for b in betas]
for b, (f, mk) in zip(betas, fit):
    print("beta = %2d  拟合指数 = %5.2f  平均连接度 = %7.1f" % (b, f, mk))
beta = next((b for b, (f, _) in zip(betas, fit) if f >= 0.8), betas[-1])
print("选定 beta =", beta)

## 3 拓扑重叠、层次聚类与切树

$\mathrm{TOM}_{ij}=\dfrac{l_{ij}+a_{ij}}{\min(k_i,k_j)+1-a_{ij}}$。以 $1-\mathrm{TOM}$ 为距离做平均连锁聚类，在固定高度 0.85 切树，少于 20 个基因的簇记为 0（相当于 WGCNA 的灰色“未归类”）。固定高度切树比 WGCNA 的动态剪切粗糙，第 6 步之后会检验结果对切树高度和 $\beta$ 是否敏感。

In [ ]:
A = ((1 + Rg) / 2) ** beta
np.fill_diagonal(A, 0.0)
k = A.sum(axis=1)
L = A @ A
TOM = (L + A) / (np.minimum.outer(k, k) + 1 - A)
np.fill_diagonal(TOM, 1.0)
D = 1 - TOM
np.fill_diagonal(D, 0.0)
tree = linkage(squareform(D, checks=False), method="average")

def cut(tree, h, min_size=20):
    c = fcluster(tree, t=h, criterion="distance")
    ids, cnt = np.unique(c, return_counts=True)
    big = [i for i, m in sorted(zip(ids, cnt), key=lambda t: -t[1]) if m >= min_size]
    return np.array([big.index(x) + 1 if x in big else 0 for x in c])

H = 0.85
mods = cut(tree, H)
sizes = pd.Series(mods).value_counts().sort_index()
print("切树高度 %.3f；模块大小（0 = 未归类）：" % H, sizes.to_dict())

## 4 模块特征基因、模块–性状关联与 hub 基因

模块特征基因（ME）取模块内标准化表达的第一主成分，符号与模块平均表达一致。ER 状态编码为 0/1，ME 与 ER 的点二列相关即皮尔逊相关。hub 用模块成员度 $k_\mathrm{ME}=\mathrm{cor}(x_i,\mathrm{ME})$ 排序。

In [ ]:
def eigengene(M):
    U, S, Vt = np.linalg.svd(M - M.mean(0), full_matrices=False)
    me = U[:, 0] * S[0]
    return me if np.corrcoef(me, M.mean(1))[0, 1] > 0 else -me

er = samples.loc[Z.index, "er_ihc"].map({"+": 1, "-": 0})
has_er = er.notna().values
rows = []
MEs = {}
for m in range(1, mods.max() + 1):
    idx = np.where(mods == m)[0]
    me = eigengene(Z.values[:, idx])
    MEs[m] = me
    r, pv = stats.pearsonr(me[has_er], er.values[has_er])
    kme = np.array([np.corrcoef(Z.values[:, i], me)[0, 1] for i in idx])
    top = genes[idx][np.argsort(-kme)[:6]]
    var_expl = np.linalg.svd(Z.values[:, idx] - Z.values[:, idx].mean(0), compute_uv=False)
    rows.append({"模块": m, "基因数": len(idx), "ME 解释方差": (var_expl[0] ** 2 / (var_expl ** 2).sum()),
                 "r(ME, ER)": r, "p": pv, "hub（kME 前 6）": ", ".join(top)})
tab = pd.DataFrame(rows).set_index("模块")
pd.set_option("display.width", 200)
print(tab.round({"ME 解释方差": 2, "r(ME, ER)": 2}).to_string(formatters={"p": "{:.1e}".format}))
best = tab["r(ME, ER)"].abs().idxmax()
print("\n与 ER 关联最强的模块：", best)

下面看几个熟悉的标志基因分别落在哪个模块：ER 通路相关的 *ESR1*、*GATA3*、*FOXA1*、*XBP1*，增殖相关的 *TOP2A*、*FOXM1*，免疫细胞标志 *PTPRC*，基质（成纤维细胞）标志 *COL1A1*，基底样标志 *KRT5*。

In [ ]:
markers = ["ESR1", "GATA3", "FOXA1", "XBP1", "SPDEF", "TOP2A", "FOXM1", "PTPRC", "COL1A1", "KRT5"]
gi = {g: i for i, g in enumerate(genes)}
for g in markers:
    if g in gi:
        m = mods[gi[g]]
        r_er = stats.pearsonr(Z.values[has_er, gi[g]], er.values[has_er])[0]
        print("%-7s 模块 %d   r(表达, ER) = %+.2f" % (g, m, r_er))

## 5 与零模型比较

**模块是否只是随机波动？** 零模型：对每个基因独立打乱它在样本间的顺序，保留每个基因的表达分布，破坏基因间的相关。在打乱的数据上重复相同的流程（同一 $\beta$、同一切树高度和最小模块大小），看还能找到多少模块，以及树上最低的合并高度。

**ME–ER 关联是否靠“挑最强模块”得来？** 对 ER 标签做 1000 次置换，每次取所有模块中最大的 $|r|$，得到“挑最强模块”这一做法本身的零分布。

In [ ]:
rng = np.random.default_rng(2026)

def mean_within_tom(T, lab):
    vals = []
    for m in range(1, lab.max() + 1):
        idx = np.where(lab == m)[0]
        sub = T[np.ix_(idx, idx)]
        vals.append(sub[~np.eye(len(idx), dtype=bool)].mean())
    return np.mean(vals) if vals else np.nan

obs_tom = mean_within_tom(TOM, mods)
null_tom, null_nmod = [], []
for rep in range(5):
    Zs = np.column_stack([rng.permutation(Z.values[:, j]) for j in range(p)])
    Rs = np.corrcoef(Zs.T)
    As = ((1 + Rs) / 2) ** beta
    np.fill_diagonal(As, 0.0)
    ks = As.sum(1)
    Ts = (As @ As + As) / (np.minimum.outer(ks, ks) + 1 - As)
    np.fill_diagonal(Ts, 1.0)
    Ds = 1 - Ts
    np.fill_diagonal(Ds, 0.0)
    ts = linkage(squareform(Ds, checks=False), method="average")
    ms = cut(ts, H)
    null_nmod.append(int(ms.max()))
    null_tom.append(ts[:, 2].min())
print("真实数据：%d 个模块，模块内平均 TOM = %.3f，树上最低合并高度 = %.3f" % (mods.max(), obs_tom, tree[:, 2].min()))
print("打乱数据（5 次）：模块数", null_nmod, "，最低合并高度", np.round(null_tom, 3))

ME_mat = np.column_stack([MEs[m][has_er] for m in MEs])
y = er.values[has_er]
obs_max = np.abs([stats.pearsonr(ME_mat[:, j], y)[0] for j in range(ME_mat.shape[1])]).max()
perm_max = np.array([np.abs([np.corrcoef(ME_mat[:, j], rng.permutation(y))[0, 1]
                             for j in range(ME_mat.shape[1])]).max() for _ in range(1000)])
print("最强模块 |r| = %.2f；置换零分布 95%% 分位数 = %.2f；置换 p < %.3f"
      % (obs_max, np.quantile(perm_max, 0.95), max((perm_max >= obs_max).mean(), 1 / 1000)))

**共表达边与物理相互作用的重叠。** 在这 1000 个基因中，把 STRING 物理相互作用（分数 ≥ 700）当作“参照边”，用 AUPR 比较三种打分：$|r|$、有符号邻接 $a_{ij}$（与 $r$ 同序）和 Schäfer–Strimmer 收缩偏相关的绝对值。随机打分的 AUPR 等于参照边比例 $\pi$。注意 STRING 物理子网络不含共表达通道，但它本身不完整，而且偏向研究充分的蛋白，所以这只是“与已知物理相互作用的一致程度”，不是推断的准确率。

In [ ]:
ppi = pd.read_csv("data/string_human_physical_v12_s700.csv")
gset = set(genes)
ppi = ppi[ppi.protein1.isin(gset) & ppi.protein2.isin(gset)]
truth = np.zeros((p, p), dtype=bool)
for a, b in zip(ppi.protein1, ppi.protein2):
    truth[gi[a], gi[b]] = truth[gi[b], gi[a]] = True
iu = np.triu_indices(p, 1)
y_true = truth[iu]
pi = y_true.mean()

def shrink_pcor(Zm):
    nn = Zm.shape[0]
    Rm = np.corrcoef(Zm.T)
    zc = (Zm - Zm.mean(0)) / Zm.std(0, ddof=1)
    var_r = np.zeros_like(Rm)
    for i in range(Zm.shape[1]):             # 逐行计算，避免 n × p × p 的大数组
        w = zc[:, [i]] * zc                  # n × p
        var_r[i] = nn / (nn - 1) ** 3 * ((w - w.mean(0)) ** 2).sum(0)
    off = ~np.eye(Zm.shape[1], dtype=bool)
    lam = min(1.0, max(0.0, var_r[off].sum() / (Rm[off] ** 2).sum()))
    Om = np.linalg.inv(lam * np.eye(Zm.shape[1]) + (1 - lam) * Rm)
    d = np.sqrt(np.diag(Om))
    return -Om / np.outer(d, d), lam

def aupr(score, y):
    o = np.argsort(-score, kind="stable")
    yy = y[o]
    tp = np.cumsum(yy)
    prec = tp / np.arange(1, len(yy) + 1)
    return prec[yy].mean()

def early_precision(score, y):
    kk = y.sum()
    return y[np.argsort(-score, kind="stable")[:kk]].mean()

P, lam = shrink_pcor(Z.values)
scores = {"|r|": np.abs(Rg)[iu], "有符号邻接 a_ij": A[iu], "|收缩偏相关|": np.abs(P)[iu]}
print("候选基因对 %d 个，参照边 %d 条，pi = %.4f；收缩强度 lambda* = %.2f" % (len(y_true), y_true.sum(), pi, lam))
res = {}
for name, s in scores.items():
    res[name] = (aupr(s, y_true), early_precision(s, y_true))
    print("%-16s AUPR = %.4f（%.1f 倍随机）  早期精度 = %.3f" % (name, res[name][0], res[name][0] / pi, res[name][1]))

## 6 作图

左：选 $\beta$ 的无标度拟合曲线；中：与 ER 关联最强的模块，ME 在 ER 阳性与阴性样本中的分布；右：三种打分前 $k$ 条边中“已知物理相互作用”的比例（精确率曲线），虚线为随机水平 $\pi$。

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
ax[0].plot(betas, [f for f, _ in fit], "o-", color="#3b6ea8")
ax[0].axhline(0.8, color="#c0392b", ls="--", lw=1)
ax[0].axvline(beta, color="grey", ls=":", lw=1)
ax[0].set_xlabel("软阈值幂次 β")
ax[0].set_ylabel("无标度拟合指数")
ax[0].set_title("选 β：首次达到 0.8 的最小幂次")

me = MEs[best][has_er]
ax[1].boxplot([me[y == 0], me[y == 1]], widths=0.5)
ax[1].set_xticks([1, 2])
ax[1].set_xticklabels(["ER−", "ER+"])
for j, grp in enumerate([me[y == 0], me[y == 1]]):
    ax[1].scatter(np.full(len(grp), j + 1) + rng.uniform(-0.12, 0.12, len(grp)), grp, s=9, alpha=0.6, color="#3b6ea8")
ax[1].set_ylabel("模块特征基因（ME）")
ax[1].set_title("模块 %d 的 ME 随 ER 状态明显不同（r = %.2f）" % (best, tab.loc[best, "r(ME, ER)"]))

kmax = 2000
for name, s in scores.items():
    yy = y_true[np.argsort(-s, kind="stable")][:kmax]
    ax[2].plot(np.arange(1, kmax + 1), np.cumsum(yy) / np.arange(1, kmax + 1), label=name)
ax[2].axhline(pi, color="grey", ls="--", lw=1, label="随机 π")
ax[2].set_xscale("log")
ax[2].set_xlabel("按得分排名的前 k 个基因对")
ax[2].set_ylabel("其中属于 STRING 物理相互作用的比例")
ax[2].set_title("排名最前的共表达边富集已知相互作用")
ax[2].legend(fontsize=8)
plt.tight_layout()

**敏感性：模块对切树高度和 $\beta$ 敏感吗？** 在 $\beta\in\{4,6,8\}$ 与切树高度 $\{0.80,0.85,0.90\}$ 的 9 种组合下重新划分，看主分析中的 ER 模块能否找回：对每种组合，计算新划分中与它重叠最大的模块的 Jaccard 指数（1 为完全相同，0 为该模块已不存在）。

In [ ]:
best_set = set(np.where(mods == best)[0])
for b2 in (beta - 2, beta, beta + 2):
    A2 = ((1 + Rg) / 2) ** b2
    np.fill_diagonal(A2, 0.0)
    k2 = A2.sum(1)
    T2 = (A2 @ A2 + A2) / (np.minimum.outer(k2, k2) + 1 - A2)
    np.fill_diagonal(T2, 1.0)
    D2 = 1 - T2
    np.fill_diagonal(D2, 0.0)
    t2 = linkage(squareform(D2, checks=False), method="average")
    for qh in (0.80, 0.85, 0.90):
        m2 = cut(t2, qh)
        jac = max(len(best_set & set(np.where(m2 == c)[0])) / len(best_set | set(np.where(m2 == c)[0]))
                  for c in range(1, m2.max() + 1)) if m2.max() > 0 else 0
        print("beta = %2d  切树高度 %.2f：%2d 个模块，与主分析 ER 模块的最大 Jaccard = %.2f" % (b2, qh, m2.max(), jac))

## 7 解读与局限

**结果说明了什么**

- 1000 个高变基因中约 400 个归入 4 个模块，其余为未归类。模块内的拓扑重叠远高于打乱数据：打乱后树上最低的合并高度约 0.97，在同样的切树规则下一个模块也找不到，说明模块结构不是随机波动。
- 按 hub 基因看，4 个模块分别对应基质/成纤维细胞（*DCN*、*FBN1*、*VCAN*、*MMP2*）、髓系免疫细胞与抗原呈递（*ITGB2*、*CYBB*、*MS4A6A*、*PTPRC*）、ER 通路与管腔上皮（*MLPH*、*FOXA1*、*XBP1*、*AGR2*、*ESR1*）以及免疫球蛋白（B 细胞/浆细胞）。其中三个模块更像是在测量活检中各类细胞的比例，而不是某条调控程序。
- 与 ER 关联最强的是 ER 通路模块（ME 与 ER 的 $r\approx0.47$），“挑最强模块”的置换检验也远超零分布。这与乳腺癌分子分型的已知结果一致：ER 状态是乳腺癌转录组中最强的变异来源之一。增殖基因（*TOP2A*、*FOXM1* 等）在这 1000 个基因里只有 8 个，没有形成达到 20 个基因的模块。
- 排名靠前的共表达边中，已知物理相互作用的比例是随机的二十多倍，但绝对比例仍然很低（前 509 对中不到 10%）：大部分强共表达基因对并没有物理相互作用。共表达说明“一起变化”，不说明“结合”或“调控”。收缩偏相关在这里反而低于 $|r|$，见下面的局限。

**局限**

- **模块由什么驱动**：组织活检的模块往往反映细胞组成（增殖细胞、免疫浸润、基质比例），见 [解读共表达模块时要追问的三件事](gene-network-inference.module-interpretation.html)。ER 模块的 hub 是 ER 状态的好**标志物**，不等于上游调控者。
- **切树方式**：敏感性分析显示，固定高度切树对 $\beta$ 很敏感。$\beta$ 改变后 TOM 的整体尺度随之改变，同一高度对应的模块粒度完全不同，ER 模块可能并入大模块或被拆散而低于 20 个基因。实际分析应在每个 $\beta$ 下重新选高度，或使用动态剪切，并报告模块对这些选择的稳定性（见 [研究问题、可复现分析与敏感性分析](reproducible-research.sensitivity-analysis.html)）。
- **基因预筛**：只用了 1000 个高变基因，模块大小与 $\beta$ 都依赖这一筛选；完整分析应在全部表达基因上进行。
- **单一队列**：模块是否在独立队列中重现，需要做 [模块保守性检验](gene-network-inference.module-preservation.html)。
- **参照网络的偏倚**：STRING 物理相互作用不完整，且偏向研究充分的蛋白。偏相关在 $n \approx 100$、$p = 1000$ 时只能依靠收缩估计；它去掉的是“经由其他基因的间接相关”，而同一细胞类型的标志基因恰恰因为共同随细胞比例变化而高度相关，偏相关会把它们之间的边削弱。这与 [偏相关与收缩估计](gene-network-inference.partial-correlation-shrinkage.html) 中“样本少时偏相关不一定优于相关排序”的结论一致，两者 AUPR 的高低不应被泛化为方法优劣。